# RouteHunter

In [1]:
import os
import pandas as pd

from routehunter import RouteHunterApp
from routehunter.utils import download_app_data

In [2]:
app_data_dir = download_app_data(to="app_data")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]

In [3]:
app = RouteHunterApp.from_data_dir(app_data_dir)

[10:31:17] Invalid InChI prefix in generating InChI Key


### 1. Review

In [4]:
print(app.review())

RouteHunter: A system for the collection and distribution of reference information on chemical synthesis routes

  Search      : check whether a given target molecule already has a known route
                either a paper reporting it, or a CASP tool that has already predicted a route for it
                
  Predict     : predict the solvability of a molecule - the chance that it can be solved by open-source CASP tools
                
  Monitor     : ranks papers by the predicted probability that they describe a multi-step synthesis route,
                based only on the paper's title and abstract.


RouteHunter data review:
  Targets                    : 1362
  Papers                     : 1263
  Targets with >1 paper      : 97
  Cached CASP routes         : 0
  Predicted candidate papers : 122865 (awaiting for digitalization)
  Papers by journal:
    Organic Process Research & Development   1240
    European Journal of Organic Chemistry    3
    Tetrahedron                    

### 2. Search

Search lets you check whether a given target molecule already has a known route - either a paper reporting it, or a CASP tool that has already predicted a route for it.

Try some molecules with positive search:  
``C#CCOC1=C(C=C(C(=C1)N2C(=O)N3CCCCC3=N2)Cl)Cl``  
``C(O)(C(O)=O)C(C1C=CC=CC=1)NC(C1C=CC=CC=1)=O``  
``C1CCC(=C(C1)CC(=O)O)N2C(=O)C=CC(=N2)C3=C4C=CC=CN4N=C3C5=CC=CC=C5``

Try some absent molecules:  
``CC(C)Cc1ccc(cc1)C(C)C(=O)O``

In [5]:
result = app.search("C(O)(C(O)=O)C(C1C=CC=CC=1)NC(C1C=CC=CC=1)=O")

In [6]:
print(result.paper_message)
result.paper_report

Found 1 paper(s) reporting a route for this molecule


,journal,title,year,doi
0,Organic Process Research & Development,Utilization of a Benzoyl Migration To Effect a...,1997,10.1021/op970113b


In [7]:
print(result.tool_message)
result.tool_report

Found 2 tool(s) predicted routes for this molecule


,tool,result,route
0,AiZynthFinder,Solved by AiZynthFinder,Cached predicted routes are not available yet
1,SynPlanner,Solved by SynPlanner,Cached predicted routes are not available yet


## 3. Predict

Predict can predict the solvability of a molecule - the chance that it can be solved by open-source CASP tools

In [8]:
result = app.predict("C(O)(C(O)=O)C(C1C=CC=CC=1)NC(C1C=CC=CC=1)=O")
print(result.to_dataframe().to_string())

            tool probability                                                             url
0  AiZynthFinder         84%                    https://github.com/MolecularAI/aizynthfinder
1     SynPlanner         82%  https://github.com/Laboratoire-de-Chemoinformatique/SynPlanner


In [9]:
result.to_dataframe()

,tool,probability,url
0,AiZynthFinder,84%,https://github.com/MolecularAI/aizynthfinder
1,SynPlanner,82%,https://github.com/Laboratoire-de-Chemoinforma...


### 4. Monitor

Monitor ranks papers by the predicted probability that they describe a multi-step synthesis route, based only on the paper's title and abstract

In [10]:
result = app.monitor(year_min=1995, year_max=2025)
result

,journal,title,abstract,doi,publication_date,route_prob
1,Organic Process Research & Development,Practical Synthesis of a HIV Integrase Inhibitor,A practical and efficient synthesis of the pot...,10.1021/op800153y,2008-10-29,0.829994
2,Tetrahedron,An expeditious route to the synthesis of adeno...,NaN,10.1016/0040-4039(96)00632-6,1996-05-01,0.818243
3,Organic Process Research & Development,Development of a Scalable Route to the SMO Rec...,A practical and scalable route to the SMO rece...,10.1021/op300170q,2012-10-31,0.814322
6,Tetrahedron,An efficient route for synthesis of spirocycli...,NaN,10.1016/j.tetlet.2024.155250,2024-08-14,0.807527
8,Organic Process Research & Development,"Convergent, Fit-For-Purpose, Kilogram-Scale Sy...",Process research and development of a syntheti...,10.1021/op200299p,2012-01-05,0.806385
...,...,...,...,...,...,...
19495,Synlett,Extending the Utility of the Bartoli Indolizat...,A short synthesis of marinoquinolines C and E ...,10.1055/s-0032-1318137,2013-01-23,0.577577
19496,Angewandte Chemie International Edition,Catalytic Asymmetric Total Synthesis of <i>ent...,Key to success: The first catalytic asymmetric...,10.1002/anie.200906678,2010-01-08,0.577574
19497,Journal of Organic Chemistry,Modular and Stereodivergent Approach to Unbran...,An iterative strategy for the stereodivergent ...,10.1021/acs.joc.6b01051,2016-08-26,0.577573
19498,Organic Letters,Asymmetric Total Synthesis of (−)-Spirofungin ...,[chemical reaction: see text]. The stereocontr...,10.1021/ol052039k,2005-11-09,0.577572


### 5. Download

Here one can download RouteHunter underlying data files, each useful for a different purpose.

In [11]:
config_df = pd.read_csv(os.path.join(app_data_dir, "config.csv"))
config_df

,key,path,comment
0,TargetStaticData,static/target_static_data.csv,Digitalized collection of targets
1,AizynthfinderStaticData,static/aizynthfinder_static_data.csv,AiZynthFinder solved-by-tool table
2,SynplannerStaticData,static/synplanner_static_data.csv,SynPlanner solved-by-tool table
3,MonitorStaticData,static/monitor_static_data.csv,High-confidence paper with route candidates
4,CandidateStaticData,static/candidate_static_data.csv,Medium-confidence paper with route candidates
5,AbstractTrainingData,build/abstract_training_data.csv,Training data for paper classifier
6,AizynthfinderPredictModel,model/aizynthfinder_predict_model.pickle,AiZynthFinder solvability model
7,SynplannerPredictModel,model/synplanner_predict_model.pickle,SynPlanner solvability model
8,PaperPredictModel,model/paper_predict_model.pickle,Paper-with-route classifier model
